In [4]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from datetime import datetime, timedelta

from core.ml.price.backtest import BacktestEngine
from core.portfolio.asset import Portfolio, Asset
from core.data.instrument.provider import InstrumentProvider
from core.data.instrument.instrument import Instrument
from core.ml.price.model import PriceDirPredictorRF, PriceDirPredictorLGBM, PriceDirPredictorXGB, PriceDirPredictorEnsemble
from core.ml.price.strategy import ATRStopLossStrategy

ip = InstrumentProvider()
instrument: Instrument = ip.get_instrument('PKO.WA')

predictor = PriceDirPredictorLGBM(instrument=instrument, horizon_days=5)
strategy = ATRStopLossStrategy(predictor)

portfolio = Portfolio('BT1', 'USD')
start_date = datetime(2025, 1, 1)
engine = BacktestEngine(portfolio, start_date, strategy, initial_cash=1000.0)


print("--- Backtest ---")
reached_present = False
i = 0
while not reached_present:
    reached_present = engine.next_day()
    print(f"Day {i+1} ({engine.date.strftime('%Y-%m-%d')}): Assets Value: ${engine.assets_values:,.2f}, Cash: ${engine.cash}, Total Value: ${engine.total_value}")
    i += 1

engine.show_transaction_log_df()

--- Backtest ---
Day 1 (2025-01-02): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 2 (2025-01-03): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 3 (2025-01-04): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 4 (2025-01-05): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 5 (2025-01-06): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 6 (2025-01-07): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 7 (2025-01-08): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 8 (2025-01-09): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 9 (2025-01-10): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 10 (2025-01-11): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 11 (2025-01-12): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 12 (2025-01-13): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 13 (2025-01-14): Assets Value: $0.00, Cash: $1000.0,

Type,Symbol,Volume,Price ($),Total Amount ($),Fee ($)
BUY,PKO.WA,15.182653,19.09,289.81,0.00
SELL,PKO.WA,15.182653,15.94,242.00,0.00
BUY,PKO.WA,9.174681,16.57,151.99,0.00
SELL,PKO.WA,9.174681,17.87,163.99,0.00
BUY,PKO.WA,11.583948,17.87,206.99,0.00
SELL,PKO.WA,11.583948,20.60,238.65,0.00
BUY,PKO.WA,11.537880,19.87,229.29,0.00
SELL,PKO.WA,11.537880,18.27,210.82,0.00
BUY,PKO.WA,9.990757,23.13,231.08,0.00
SELL,PKO.WA,9.990757,26.49,264.65,0.00


In [5]:
portfolio2 = Portfolio('BT2', 'USD')
start_date = datetime(2025, 1, 1)
market_data = instrument.get_market_data_at_closest_trading_day(start_date)
price = portfolio2.convert_to_native_currency(market_data['close'], instrument.currency)

volume = 1000 / price if price != 0 else 0

asset = Asset(instrument, volume, price, start_date)

portfolio2.add(asset)

portfolio2.value

1947.2361809045226